# Tabular Machine Learning Models for Walmart Sales Forecasting

This notebook trains, evaluates, and compares nine different machine learning models and neural networks on the Walmart sales forecasting task. We use the tabular features engineered in previous steps and evaluate all models using the standard validation split to ensure direct comparison with our baseline and statistical models.

## Models Implemented:
1. **Linear Regression** (OLS)
2. **Random Forest Regressor** (Bagging ensemble of decision trees)
3. **K-Nearest Neighbors (KNN)**
4. **XGBoost Regressor** (Gradient boosted trees with regularization)
5. **LightGBM Regressor** (Leaf-wise histogram gradient boosted trees)
6. **Multi-Layer Perceptron (MLP)** (Scikit-Learn Multi-Layer Perceptron)
7. **Artificial Neural Network (ANN)** (PyTorch feed-forward network with linear & ReLU layers)
8. **Gradient Boosted Regression Trees (GBRT)** (Scikit-Learn Gradient Boosting Regressor)
9. **Ensemble Regressor** (Simple average of Random Forest + XGBoost + LightGBM + MLP predictions)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Append project root
sys.path.append(os.path.abspath(".."))

from src import config
from src.evaluate_models import (
    get_train_val_split,
    evaluate_predictions,
    calculate_wmae,
    calculate_mae
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

np.random.seed(42)
torch.manual_seed(42)

## 1. Load and Clean Tabular Features

We load the engineered features from `data/processed/train_features.csv` and drop rows containing NaNs (resulting from initial lag offsets).

In [ ]:
# Load engineered features
features_path = Path("..") / config.TRAIN_FEATURES_PATH.relative_to(config.BASE_DIR)
df = pd.read_csv(features_path)

# Standardize IsHoliday to boolean
if df["IsHoliday"].dtype == "object":
    df["IsHoliday"] = (
        df["IsHoliday"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )
df["IsHoliday"] = df["IsHoliday"].astype(bool)

df["Date"] = pd.to_datetime(df["Date"])

# Define features list
feature_cols = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price',
    'CPI', 'Unemployment', 'Year', 'Month', 'WeekOfYear', 'Quarter',
    'Is_Q4', 'Is_Holiday_Month', 'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
    'MarkDown_Total', 'Has_MarkDown', 'MarkDown_Avg', 'MarkDown_Max',
    'Weekly_Sales_lag_1', 'Weekly_Sales_lag_2', 'Weekly_Sales_lag_3', 'Weekly_Sales_lag_4',
    'Weekly_Sales_lag_8', 'Weekly_Sales_lag_12', 'Weekly_Sales_lag_26', 'Weekly_Sales_lag_52',
    'rolling_mean_4', 'rolling_mean_12', 'rolling_mean_26', 'rolling_mean_52',
    'rolling_std_4', 'rolling_std_12', 'rolling_std_52',
    'Store_Avg_Sales', 'Dept_Avg_Sales', 'Store_Dept_Avg_Sales',
    'Type_Encoded', 'Holiday_MarkDown', 'Q4_Holiday', 'Size_MarkDown'
]

# Drop rows with missing values in target or features
df_clean = df.dropna(subset=['Weekly_Sales'] + feature_cols).copy()

print(f"Original rows: {len(df):,}")
print(f"Cleaned rows:  {len(df_clean):,}")

## 2. Train / Validation Split & Scaling

We split the cleaned dataset using the standard train/validation split date `2012-08-10`.

In [ ]:
# Split into train and validation sets
train_df = df_clean[df_clean["Date"] <= "2012-08-10"].copy()
val_df = df_clean[df_clean["Date"] > "2012-08-10"].copy()

X_train = train_df[feature_cols]
y_train = train_df["Weekly_Sales"]
X_val = val_df[feature_cols]
y_val = val_df["Weekly_Sales"]

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")

# Standard Scaling for linear/distance-based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## 3. Train Machine Learning Models

We define and train the baseline and advanced machine learning models.

In [ ]:
results_dict = {}
metrics_list = []

def log_metrics(name, preds, runtime):
    val_df[f"{name}_Pred"] = preds
    metrics = evaluate_predictions(
        y_true=y_val,
        y_pred=preds,
        is_holiday=val_df["IsHoliday"]
    )
    metrics["Model"] = name
    metrics["Runtime (s)"] = runtime
    metrics_list.append(metrics)
    print(f"  WMAE: {metrics['WMAE']:,.2f} | MAE: {metrics['MAE']:,.2f} | Runtime: {runtime:.2f}s")

# --- 1. Linear Regression ---
print("Training Linear Regression...")
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
preds_lr = lr.predict(X_val_scaled)
log_metrics("Linear Regression", preds_lr, time.time() - t0)

# --- 2. Random Forest ---
print("\nTraining Random Forest...")
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_val)
log_metrics("Random Forest", preds_rf, time.time() - t0)

# --- 3. KNN ---
print("\nTraining K-Nearest Neighbors (KNN)...")
t0 = time.time()
knn = KNeighborsRegressor(n_neighbors=5, weights='distance', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
preds_knn = knn.predict(X_val_scaled)
log_metrics("KNN", preds_knn, time.time() - t0)

# --- 4. XGBoost ---
print("\nTraining XGBoost...")
t0 = time.time()
xgb = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_val)
log_metrics("XGBoost", preds_xgb, time.time() - t0)

# --- 5. LightGBM ---
print("\nTraining LightGBM...")
t0 = time.time()
lgb = LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
lgb.fit(X_train, y_train)
preds_lgb = lgb.predict(X_val)
log_metrics("LightGBM", preds_lgb, time.time() - t0)

# --- 6. MLP (Scikit-Learn Multi-Layer Perceptron) ---
print("\nTraining MLP...")
t0 = time.time()
mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=20, random_state=42, batch_size=256, early_stopping=True)
mlp.fit(X_train_scaled, y_train)
preds_mlp = mlp.predict(X_val_scaled)
log_metrics("MLP", preds_mlp, time.time() - t0)

# --- 7. ANN (PyTorch Artificial Neural Network) ---
print("\nTraining ANN (PyTorch)...")
t0 = time.time()
class ANNRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=1024, shuffle=True)
ann = ANNRegressor(X_train_scaled.shape[1])
optimizer = torch.optim.Adam(ann.parameters(), lr=0.01)
criterion = nn.MSELoss()

ann.train()
for epoch in range(10):
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = ann(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()

ann.eval()
with torch.no_grad():
    preds_ann = ann(X_val_t).numpy()
log_metrics("ANN", preds_ann, time.time() - t0)

# --- 8. GBRT (Gradient Boosted Regression Trees) ---
print("\nTraining GBRT (Scikit-Learn)...")
t0 = time.time()
gbrt = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
gbrt.fit(X_train, y_train)
preds_gbrt = gbrt.predict(X_val)
log_metrics("GBRT", preds_gbrt, time.time() - t0)

# --- 9. Ensemble Learning (RF + XGBoost + LightGBM + MLP) ---
print("\nCreating Ensemble Predictions...")
t0 = time.time()
preds_ensemble = (preds_rf + preds_xgb + preds_lgb + preds_mlp) / 4.0
log_metrics("Ensemble", preds_ensemble, time.time() - t0)

## 4. Compare Model Error Metrics

We compile error metrics across all models and display them in a ranked summary table.

In [ ]:
metrics_df = pd.DataFrame(metrics_list)[["Model", "WMAE", "MAE", "RMSE", "MAPE", "sMAPE", "Runtime (s)"]]
metrics_df = metrics_df.sort_values("WMAE").reset_index(drop=True)
display(metrics_df)

## 5. Visualizing Model Error Comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_plot = ["WMAE", "MAE", "RMSE"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for idx, metric in enumerate(metrics_plot):
    sns.barplot(data=metrics_df, x="Model", y=metric, ax=axes[idx], color=colors[idx])
    axes[idx].set_title(f"{metric} Comparison", fontsize=13, fontweight="bold")
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].set_ylabel(metric)
    axes[idx].set_xlabel("")
    for p in axes[idx].patches:
        height = p.get_height()
        axes[idx].annotate(f'{height:,.1f}',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.suptitle("Machine Learning Models Error Comparison (All Groups)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Actual vs Predicted Weekly Sales

We plot predicted vs actual sales across a subset of validation dates to visualize alignment.

In [ ]:
agg_sales = val_df.groupby("Date")["Weekly_Sales"].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(agg_sales["Date"], agg_sales["Weekly_Sales"], label="Actual Sales", color="black", linewidth=3.0, marker="o")

top_models = metrics_df.head(3)["Model"].tolist()
for name in top_models:
    col_name = f"{name}_Pred"
    pred_agg = val_df.groupby("Date")[col_name].sum().reset_index()
    plt.plot(pred_agg["Date"], pred_agg[col_name], label=name, linestyle="--", marker="x", alpha=0.8)

plt.title("Aggregate Weekly Forecast vs Actual Sales (Top 3 Machine Learning Models)", fontsize=15, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Sales ($)", fontsize=12)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

## 7. Actual vs Predicted Scatter Analysis

We construct scatter plots of predicted vs actual sales capped at the 99th percentile of sales values.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

plot_limit = max(
    val_df["Weekly_Sales"].quantile(0.99),
    val_df[[f"{name}_Pred" for name in models_keys := list(models_keys_list := [m["Model"] for m in metrics_list])]].quantile(0.99).max()
)

for idx, name in enumerate(models_keys):
    col_name = f"{name}_Pred"
    ax = axes[idx]
    
    ax.scatter(val_df["Weekly_Sales"], val_df[col_name], alpha=0.3, color="#4c72b0", edgecolors="w")
    ax.plot([0, plot_limit], [0, plot_limit], color="red", linestyle="--", linewidth=2)
    ax.set_xlim(0, plot_limit)
    ax.set_ylim(0, plot_limit)
    
    corr = val_df["Weekly_Sales"].corr(val_df[col_name])
    model_wmae = metrics_df.set_index("Model").loc[name, "WMAE"]
    
    ax.set_title(f"{name}\nr = {corr:.3f}, WMAE = {model_wmae:,.0f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Actual Weekly Sales ($)")
    ax.set_ylabel("Predicted Weekly Sales ($)")
    
plt.suptitle("Actual vs Predicted Weekly Sales Scatter Analysis (Capped at 99th Percentile)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Save Model Predictions and Metrics

We serialize the validation set predictions and output metrics to files.

In [ ]:
import joblib
import json
from pathlib import Path

# 1. Save Trained Machine Learning Models to models/
models_dir = PROJECT_ROOT / "models" if "PROJECT_ROOT" in globals() else Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

if "models" in globals() and isinstance(models, dict):
    for m_name, model_obj in models.items():
        safe_name = m_name.lower().replace(" ", "_").replace("-", "_").replace("+", "")
        save_model_path = models_dir / f"ml_{safe_name}.joblib"
        joblib.dump(model_obj, save_model_path)
        print(f"Saved ML model checkpoint to: {save_model_path}")

# 2. Save Predictions & Metrics to results/
output_dir = PROJECT_ROOT / "results" if "PROJECT_ROOT" in globals() else Path("../results")
output_dir.mkdir(parents=True, exist_ok=True)

pred_path = output_dir / "ml_predictions.csv"
val_df.to_csv(pred_path, index=False)
print(f"Predictions saved successfully to: {pred_path.resolve()}")

metrics_path = output_dir / "ml_metrics.json"
metrics_dict_save = metrics_df.set_index("Model").to_dict(orient="index")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_dict_save, f, indent=4)
print(f"Metrics saved successfully to: {metrics_path.resolve()}")

## 9. Key Findings & Conclusions

In [ ]:
best_row = metrics_df.iloc[0]
print("=== SUMMARY OF KEY FINDINGS ===")
print(f"Best Performing Machine Learning Model: {best_row['Model']}")
print(f"Validation WMAE:                        {best_row['WMAE']:,.2f}")